# Processamento de Linguagem Natural
## Exercícios TC.10 e PC.10

**Disciplina:** Processamento de Linguagem Natural
**Professor:** Fabrício Galende Marques de Carvalho
**Aluno(a):** _(preencher)_
**Data:** 24 de agosto de 2026

Este notebook contém o código-fonte executável referente à demonstração do exercício **TC.10** e à resolução completa do exercício **PC.10**. A discussão conceitual detalhada de cada exercício está disponível na apresentação em PDF que acompanha este material.

### Procedimento de execução

1. Criar e ativar um ambiente virtual Python 3 (recomendado):

   ```bash
   python3 -m venv venv
   source venv/bin/activate        # Linux/macOS
   venv\Scripts\activate           # Windows
   ```

2. Instalar as dependências listadas em `requirements.txt` (o arquivo já inclui o modelo `pt_core_news_sm` do spaCy):

   ```bash
   pip install -r requirements.txt
   ```

3. Abrir este notebook com o Jupyter:

   ```bash
   jupyter notebook TC10_PC10_PLN.ipynb
   ```

4. Executar as células em sequência (`Run All`). A função `assistente_virtual()` (seção PC.10) inicia um loop interativo via `input()` e está, por isso, comentada na célula de definição — para testá-la manualmente, descomente a chamada e execute a célula individualmente, pois o `input()` bloquearia a execução automática de "Run All".

## TC.10 — Emojis e símbolos não formais em sistemas de PLN

**Enunciado:** Por que a utilização de emojis ou outros símbolos não presentes na linguagem textual formal podem dificultar a operação de um sistema de PLN? Cite um exemplo prático de sentença textual comumente presente em diálogos que faz uso de emojis e ilustre qual a dificuldade pode ser gerada ao sistema de PLN.

Este exercício é essencialmente teórico e não exige, a rigor, o desenvolvimento de um programa. Ainda assim, a célula de código a seguir demonstra na prática, com uma frase real contendo emojis, o problema discutido em detalhes na apresentação em PDF: os modelos de PLN são treinados sobre vocabulário linguístico formal e não possuem representação adequada para símbolos pictográficos.

In [1]:
import spacy
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

nlp = spacy.load("pt_core_news_sm")

In [2]:
frase = "Adorei o produto \U0001F60D\U0001F44F, mas a entrega atrasou muito \U0001F621"

print("Frase de entrada:")
print(frase)
print()

doc = nlp(frase)
print(f"{'Token':<10}{'POS':<8}{'Lema':<10}")
print("-" * 28)
for token in doc:
    print(f"{token.text:<10}{token.pos_:<8}{token.lemma_:<10}")

print()
print("Tokenização com NLTK (word_tokenize):")
print(word_tokenize(frase, language="portuguese"))

Frase de entrada:
Adorei o produto 😍👏, mas a entrega atrasou muito 😡

Token     POS     Lema      
----------------------------
Adorei    VERB    Adorei    
o         DET     o         
produto   NOUN    produto   
😍         ADJ     😍         
👏         NOUN    👏         
,         PUNCT   ,         
mas       CCONJ   mas       
a         DET     o         
entrega   NOUN    entrega   
atrasou   VERB    atrasar   
muito     ADV     muito     
😡         VERB    😡         

Tokenização com NLTK (word_tokenize):
['Adorei', 'o', 'produto', '😍👏', ',', 'mas', 'a', 'entrega', 'atrasou', 'muito', '😡']


### Discussão do resultado

Observando a saída acima, é possível notar dois problemas concretos:

1. **Classificação gramatical (POS) incoerente:** o spaCy atribui aos emojis 😍, 👏 e 😡 as classes `ADJ`, `NOUN` e `VERB`, respectivamente — rótulos sem qualquer sentido linguístico real, pois o modelo estatístico nunca observou esses símbolos durante o treinamento e faz uma estimativa forçada a partir do contexto. O lema atribuído a cada emoji é o próprio símbolo, ou seja, nenhuma informação semântica é efetivamente extraída.
2. **Divergência de tokenização:** o tokenizador do spaCy separa 😍 e 👏 em dois tokens distintos, enquanto o `word_tokenize` do NLTK mantém os dois símbolos agrupados em um único token (`'😍👏'`). Isso mostra que ferramentas diferentes de PLN não possuem um tratamento padronizado para esse tipo de símbolo, o que compromete a reprodutibilidade e a comparabilidade dos resultados entre pipelines distintas.

Além dos aspectos técnicos observados, há um problema semântico relevante: os emojis carregam **carga emocional e de sentimento** que é central para tarefas como análise de sentimentos. Na frase de exemplo, a presença de 😍 e 👏 reforça uma opinião positiva sobre o produto, enquanto 😡 reforça uma opinião negativa sobre a entrega. Um sistema de PLN que ignore os emojis (tratando-os como ruído a ser removido no pré-processamento) perde essa informação, podendo classificar erroneamente o sentimento geral da frase. Já um sistema que os mantenha sem tratamento especializado, como demonstrado acima, obtém rótulos gramaticais sem sentido, que podem prejudicar etapas posteriores da pipeline, como a extração de características ou a criação do modelo de linguagem.

A solução usual, detalhada na apresentação em PDF, envolve uma etapa específica de pré-processamento, na qual os emojis são convertidos para tokens textuais equivalentes (e.g., 😍 → `"carinha_apaixonada"` ou `"sentimento_positivo"`) por meio de bibliotecas especializadas (e.g., `emoji`, `demoji`), preservando a informação emocional antes das demais etapas da pipeline de PLN.

## PC.10 — Assistente virtual baseado em tags POS

**Enunciado:** Fazendo uso de tags do tipo POS, desenvolva um assistente virtual simples que identifique se o usuário está fazendo uma pergunta ou se é afirmação ou comando. *Hint: olhe para a primeira palavra da frase e seu POS.*

A estratégia adotada segue a dica do enunciado: a classificação é feita observando a classe gramatical (POS) da primeira palavra da frase, com o apoio de regras linguísticas simples, descritas a seguir:

- **Comando:** a frase inicia com um verbo. No português, o imperativo afirmativo dirigido a "você"/"ele/ela" é morfologicamente idêntico ao presente do subjuntivo (e.g., "Feche a porta" vs. "que você feche a porta"). Por isso, tanto `Mood=Imp` quanto `Mood=Sub` no primeiro token verbal são utilizados como indício de comando.
- **Pergunta:** a frase inicia com um pronome ou advérbio interrogativo (e.g., "quem", "onde", "como", "quando", "qual"), ou termina com o sinal de interrogação `?` (necessário para perguntas do tipo sim/não, como "Você fechou a porta?", que não possuem nenhuma marca morfológica na primeira palavra).
- **Afirmação:** caso nenhuma das regras anteriores seja satisfeita, a frase é tratada como uma afirmação.

In [3]:
PALAVRAS_INTERROGATIVAS = {
    "que", "quem", "qual", "quais", "quando", "onde",
    "como", "quanto", "quantos", "quanta", "quantas",
}


def identificar_tipo_frase(frase: str) -> str:
    """Classifica uma frase em 'Pergunta', 'Comando' ou 'Afirmação'
    a partir da classe gramatical (POS) da primeira palavra."""
    frase = frase.strip()
    # Reduz o viés de maiuscula inicial dos modelos estatisticos de POS,
    # que tendem a classificar palavras capitalizadas como nomes proprios.
    frase_normalizada = frase[0].lower() + frase[1:] if frase else frase

    doc = nlp(frase_normalizada)
    primeiro = doc[0]
    pos = primeiro.pos_
    modo = primeiro.morph.get("Mood")

    if pos in ("PRON", "DET", "ADV", "SCONJ") and primeiro.text.lower() in PALAVRAS_INTERROGATIVAS:
        return "Pergunta"

    if pos in ("VERB", "AUX") and modo and modo[0] in ("Imp", "Sub"):
        return "Comando"

    if frase.endswith("?"):
        return "Pergunta"

    return "Afirmação"

In [4]:
frases_teste = [
    "Você fechou a porta?",
    "Feche a porta, por favor.",
    "O gato está dormindo no sofá.",
    "Quem chamou você ontem?",
    "Envie o relatório até amanhã.",
    "Ela chegou cedo para a reunião.",
    "Onde fica a estação mais próxima?",
    "Abra a porta.",
]

print(f"{'Frase':<38}{'Classificação'}")
print("-" * 55)
for frase in frases_teste:
    print(f"{frase:<38}{identificar_tipo_frase(frase)}")

Frase                                 Classificação
-------------------------------------------------------
Você fechou a porta?                  Pergunta
Feche a porta, por favor.             Comando
O gato está dormindo no sofá.         Afirmação
Quem chamou você ontem?               Pergunta
Envie o relatório até amanhã.         Comando
Ela chegou cedo para a reunião.       Afirmação
Onde fica a estação mais próxima?     Pergunta
Abra a porta.                         Comando


### Resultado da classificação

As 8 frases de teste acima cobrem os três tipos de sentença (pergunta com e sem pronome interrogativo, comando e afirmação) e foram todas corretamente classificadas pelas regras baseadas em POS, confirmando que a estratégia sugerida no enunciado — observar a primeira palavra e seu POS — é suficiente para o cenário testado.

In [5]:
def assistente_virtual():
    """Loop interativo simples que classifica cada frase digitada pelo usuário.
    Digite 'sair' para encerrar."""
    print("Assistente virtual iniciado. Digite 'sair' para encerrar.\n")
    while True:
        frase = input("Você: ")
        if frase.strip().lower() == "sair":
            print("Assistente: Até logo!")
            break
        tipo = identificar_tipo_frase(frase)
        print(f"Assistente: identifiquei isso como uma {tipo.lower()}.\n")


# Para testar interativamente, execute a linha abaixo em uma célula própria:
# assistente_virtual()

### Limitações observadas

O modelo `pt_core_news_sm` do spaCy é um modelo estatístico pequeno e, por isso, comete erros de classificação gramatical em alguns casos, especialmente quando a primeira palavra da frase é um verbo pouco frequente em posição inicial. O exemplo a seguir ilustra esse problema.

In [6]:
frase_problema = "Compre pão no mercado."
frase_norm = frase_problema[0].lower() + frase_problema[1:]
doc = nlp(frase_norm)
print(f"Frase: {frase_problema!r}")
print(f"Primeira palavra: '{doc[0].text}' | POS obtido: {doc[0].pos_} (esperado: VERB)")
print(f"Classificação do assistente: {identificar_tipo_frase(frase_problema)} (esperado: Comando)")

Frase: 'Compre pão no mercado.'
Primeira palavra: 'compre' | POS obtido: ADJ (esperado: VERB)
Classificação do assistente: Afirmação (esperado: Comando)


Nesse caso, o modelo classifica erroneamente "compre" como adjetivo, e a frase acaba sendo tratada como afirmação, quando deveria ser um comando. Esse tipo de erro decorre diretamente da qualidade do modelo estatístico utilizado (modelos maiores, como `pt_core_news_lg`, tendem a cometer menos erros desse tipo, ao custo de maior tempo de processamento e maior consumo de memória).

Outras limitações relevantes:

- A regra de pergunta do tipo sim/não depende exclusivamente do sinal de pontuação `?`, pois o português não possui, em geral, nenhuma marca morfológica na primeira palavra que distinga uma pergunta sim/não de uma afirmação (e.g., "Você fechou a porta?" vs. "Você fechou a porta."). Em contextos de fala transcrita sem pontuação, essa regra deixaria de funcionar corretamente.
- Frases compostas por oração subordinada antes da oração principal (e.g., "Se puder, feche a porta.") não são tratadas corretamente por este assistente simplificado, pois a primeira palavra não corresponde ao verbo principal da frase.

Essas limitações são discutidas com mais profundidade na apresentação em PDF que acompanha este notebook.